# 5. Pipeline de regresión lineal

Se construye un modelo de regresión lineal para estimar el salario mensual en quetzales utilizando seis predictores: edad, antigüedad, horas semanales, nivel educativo, categoría ocupacional y dominio.

Se utilizan los primeros tres trimestres de 2025 para entrenar y el cuarto trimestre para validar y seleccionar la regularización. El primer trimestre de 2026 se reserva para la evaluación final.

Como referencia, se utiliza un modelo que predice para todos los registros el salario promedio del conjunto de entrenamiento.

In [7]:
import os
import json
import pandas as pd

from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab7_RegresionLineal")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print("Versión de Spark:", spark.version)

assert spark.version.startswith("3.5."), "El laboratorio requiere Spark 3.5.x."

Versión de Spark: 3.5.1


26/09/25 01:54:00 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [8]:
PARQUET_DIR = "../working_dir/parquet"
RUTA_2025 = f"{PARQUET_DIR}/eneic_2025_preparado"

NUMERICAS = [
    "edad",
    "antiguedad",
    "horas_semanales"
]

CATEGORICAS = [
    "nivel_educativo",
    "categoria_ocupacional",
    "dominio"
]

OBJETIVO = "salario_mensual"

COLUMNAS_MODELO = [
    "periodo_archivo",
    "NUM_HOGAR",
    "NUM_PERSONA",
    OBJETIVO,
    *NUMERICAS,
    *CATEGORICAS
]

df_2025_modelo = (
    spark.read.parquet(RUTA_2025)
    .select(*COLUMNAS_MODELO)
)

print(f"Registros preparados de 2025: {df_2025_modelo.count():,}")

df_2025_modelo.printSchema()
df_2025_modelo.show(5, truncate=False)

Registros preparados de 2025: 53,025
root
 |-- periodo_archivo: string (nullable = true)
 |-- NUM_HOGAR: long (nullable = true)
 |-- NUM_PERSONA: integer (nullable = true)
 |-- salario_mensual: double (nullable = true)
 |-- edad: double (nullable = true)
 |-- antiguedad: double (nullable = true)
 |-- horas_semanales: double (nullable = true)
 |-- nivel_educativo: string (nullable = true)
 |-- categoria_ocupacional: string (nullable = true)
 |-- dominio: string (nullable = true)

+---------------+---------+-----------+---------------+----+-------------------+---------------+---------------+---------------------+-------+
|periodo_archivo|NUM_HOGAR|NUM_PERSONA|salario_mensual|edad|antiguedad         |horas_semanales|nivel_educativo|categoria_ocupacional|dominio|
+---------------+---------+-----------+---------------+----+-------------------+---------------+---------------+---------------------+-------+
|2025T1         |17085    |1          |5000.0         |33.0|8.0                |40.0   

## Entrenamiento y validación

In [9]:
train = (
    df_2025_modelo
    .filter(F.col("periodo_archivo").isin("2025T1", "2025T2", "2025T3"))
    .cache()
)

validacion = (
    df_2025_modelo
    .filter(F.col("periodo_archivo") == "2025T4")
    .cache()
)

n_train = train.count()
n_validacion = validacion.count()

assert n_train > 0, "El conjunto de entrenamiento está vacío."
assert n_validacion > 0, "El conjunto de validación está vacío."

print(f"Entrenamiento: {n_train:,} registros")
print(f"Validación:    {n_validacion:,} registros")

(
    train.withColumn("conjunto", F.lit("Entrenamiento"))
    .unionByName(
        validacion.withColumn("conjunto", F.lit("Validación"))
    )
    .groupBy("conjunto", "periodo_archivo")
    .count()
    .orderBy("periodo_archivo")
    .show(truncate=False)
)

Entrenamiento: 40,361 registros
Validación:    12,664 registros
+-------------+---------------+-----+
|conjunto     |periodo_archivo|count|
+-------------+---------------+-----+
|Entrenamiento|2025T1         |13419|
|Entrenamiento|2025T2         |13492|
|Entrenamiento|2025T3         |13450|
|Validación   |2025T4         |12664|
+-------------+---------------+-----+



## Preparación del pipeline y criterios de evaluación

Las variables categóricas se procesan con StringIndexer y OneHotEncoder. Esta codificación evita interpretar sus códigos como cantidades o como una escala numérica.

StringIndexer utiliza handleInvalid="keep" para conservar registros con categorías no observadas durante el entrenamiento.

VectorAssembler combina los tres predictores numéricos y las tres variables categóricas codificadas. Se utiliza la estandarización interna de LinearRegression mediante standardization=True.

Todos los componentes del pipeline se ajustan únicamente con entrenamiento.

Se calculan tres métricas sobre todos los registros de validación:

- MAE
- RMSE
- R²

La configuración se selecciona por el menor RMSE de validación. Además, se compara con una referencia que utiliza exclusivamente la media salarial del entrenamiento.

## Métricas y modelo de referencia

In [10]:
evaluadores = {
    "MAE": RegressionEvaluator(
        labelCol=OBJETIVO,
        predictionCol="prediction",
        metricName="mae"
    ),
    "RMSE": RegressionEvaluator(
        labelCol=OBJETIVO,
        predictionCol="prediction",
        metricName="rmse"
    ),
    "R2": RegressionEvaluator(
        labelCol=OBJETIVO,
        predictionCol="prediction",
        metricName="r2"
    )
}

def calcular_metricas(predicciones):
    return {
        nombre: evaluador.evaluate(predicciones)
        for nombre, evaluador in evaluadores.items()
    }

media_train = train.agg(
    F.avg(OBJETIVO).alias("media")
).first()["media"]

pred_referencia = validacion.withColumn(
    "prediction",
    F.lit(float(media_train))
)

metricas_referencia = calcular_metricas(pred_referencia)

print(f"Salario predicho por la referencia: Q{media_train:,.2f}")

display(
    pd.DataFrame([{
        "modelo": "Referencia: media de entrenamiento",
        **metricas_referencia
    }]).round(4)
)

Salario predicho por la referencia: Q3,383.12


,modelo,MAE,RMSE,R2
0,Referencia: media de entrenamiento,1672.5026,2889.5714,-0.0031


## Función para construir el Pipeline

In [11]:
def crear_pipeline_lr(reg_param, elastic_net_param):
    indexadores = [
        StringIndexer(
            inputCol=columna,
            outputCol=f"{columna}_idx",
            handleInvalid="keep",
            stringOrderType="alphabetAsc"
        )
        for columna in CATEGORICAS
    ]

    codificador = OneHotEncoder(
        inputCols=[f"{columna}_idx" for columna in CATEGORICAS],
        outputCols=[f"{columna}_ohe" for columna in CATEGORICAS],
        dropLast=True
    )

    ensamblador = VectorAssembler(
        inputCols=NUMERICAS + [
            f"{columna}_ohe" for columna in CATEGORICAS
        ],
        outputCol="features",
        handleInvalid="error"
    )

    regresion = LinearRegression(
        featuresCol="features",
        labelCol=OBJETIVO,
        predictionCol="prediction",
        standardization=True,
        regParam=reg_param,
        elasticNetParam=elastic_net_param,
        maxIter=200,
        tol=1e-6
    )

    return Pipeline(
        stages=indexadores + [
            codificador,
            ensamblador,
            regresion
        ]
    )

## Configuraciones de regularización

Se comparan tres configuraciones:

| Configuración | regParam | elasticNetParam |
|---|---:|---:|
| Sin regularización | 0.0 | 0.0 |
| Ridge | 0.1 | 0.0 |
| Elastic Net | 0.1 | 0.5 |

regParam controla la intensidad de la penalización.

Con elasticNetParam=0 se utiliza penalización L2 (Ridge), mientras que elasticNetParam=0.5 combina las penalizaciones L1 y L2.

La regularización puede reducir la magnitud de los coeficientes y mejorar la generalización. Su utilidad se evalúa con el RMSE de validación, sin asumir que una configuración regularizada será mejor.

## Entrenar y comparar configuraciones

In [12]:
configuraciones = [
    {
        "configuracion": "Sin regularización",
        "regParam": 0.0,
        "elasticNetParam": 0.0
    },
    {
        "configuracion": "Ridge",
        "regParam": 0.1,
        "elasticNetParam": 0.0
    },
    {
        "configuracion": "Elastic Net",
        "regParam": 0.1,
        "elasticNetParam": 0.5
    }
]

modelos_lr = {}
resultados_lr = []

for config in configuraciones:
    nombre = config["configuracion"]

    pipeline = crear_pipeline_lr(
        reg_param=config["regParam"],
        elastic_net_param=config["elasticNetParam"]
    )

    modelo = pipeline.fit(train)
    predicciones = modelo.transform(validacion).cache()

    try:
        assert predicciones.count() == n_validacion, (
            "El pipeline cambió el número de registros de validación."
        )

        metricas = calcular_metricas(predicciones)

        modelos_lr[nombre] = modelo
        resultados_lr.append({
            **config,
            **metricas
        })

        print(
            f"{nombre}: "
            f"MAE = Q{metricas['MAE']:,.2f} | "
            f"RMSE = Q{metricas['RMSE']:,.2f} | "
            f"R² = {metricas['R2']:.4f}"
        )
    finally:
        predicciones.unpersist()

tabla_lr = (
    pd.DataFrame(resultados_lr)
    .sort_values("RMSE", kind="stable")
    .reset_index(drop=True)
)

display(tabla_lr.round(4))

26/09/25 02:02:58 WARN Instrumentation: [36842ae7] regParam is zero, which might cause numerical instability and overfitting.
26/09/25 02:02:58 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/09/25 02:02:58 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/09/25 02:02:58 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
26/09/25 02:02:58 WARN Instrumentation: [36842ae7] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


Sin regularización: MAE = Q1,210.14 | RMSE = Q2,186.42 | R² = 0.4257
Ridge: MAE = Q1,210.13 | RMSE = Q2,186.42 | R² = 0.4257
Elastic Net: MAE = Q1,210.11 | RMSE = Q2,186.42 | R² = 0.4257


,configuracion,regParam,elasticNetParam,MAE,RMSE,R2
0,Sin regularización,0.0,0.0,1210.1375,2186.4196,0.4257
1,Ridge,0.1,0.0,1210.1327,2186.4209,0.4257
2,Elastic Net,0.1,0.5,1210.1070,2186.4248,0.4257


In [13]:
mejor_nombre = tabla_lr.loc[0, "configuracion"]
mejor_modelo_lr = modelos_lr[mejor_nombre]

mejor_config_lr = next(
    config.copy()
    for config in configuraciones
    if config["configuracion"] == mejor_nombre
)

metricas_mejor_lr = {
    metrica: float(tabla_lr.loc[0, metrica])
    for metrica in ["MAE", "RMSE", "R2"]
}

comparacion_validacion = pd.DataFrame([
    {
        "modelo": "Referencia: media de entrenamiento",
        **metricas_referencia
    },
    {
        "modelo": f"Regresión lineal: {mejor_nombre}",
        **metricas_mejor_lr
    }
])

comparacion_validacion["reduccion_RMSE_pct"] = (
    100 * (
        1
        - comparacion_validacion["RMSE"]
        / metricas_referencia["RMSE"]
    )
)

print("Mejor configuración:", mejor_nombre)
print("regParam:", mejor_config_lr["regParam"])
print("elasticNetParam:", mejor_config_lr["elasticNetParam"])

display(comparacion_validacion.round(4))

# Se conserva para posteriores análisis sobre validación.
pred_validacion_lr = mejor_modelo_lr.transform(validacion)

Mejor configuración: Sin regularización
regParam: 0.0
elasticNetParam: 0.0


,modelo,MAE,RMSE,R2,reduccion_RMSE_pct
0,Referencia: media de entrenamiento,1672.5026,2889.5714,-0.0031,0.0000
1,Regresión lineal: Sin regularización,1210.1375,2186.4196,0.4257,24.3341


### Comparación con el modelo de referencia

El modelo de referencia predice para todos los registros el salario
promedio del entrenamiento. En validación obtuvo un MAE de Q1,672.50,
un RMSE de Q2,889.57 y un R² de -0.0031.

La regresión lineal sin regularización redujo el MAE a Q1,210.14
y el RMSE a Q2,186.42. Esto representa una reducción del 27.64%
en MAE y del 24.33% en RMSE respecto a la referencia. Por tanto,
incorporar las características personales y laborales mejora
la predicción.

El R² de la regresión fue 0.4257, equivalente a explicar aproximadamente
el 42.57% de la variabilidad salarial del conjunto de validación.
Sin embargo, los errores todavía son considerables.

El R² ligeramente negativo de la referencia se explica porque utiliza
la media del entrenamiento, mientras que el R² toma como referencia
la media del conjunto de validación. 

Se seleccionó la configuración sin regularización por presentar
el menor RMSE de validación. El desempeño similar entre entrenamiento y validación no muestra
una brecha marcada de sobreajuste.

Los resultados describen los registros analizados, no estimaciones
oficiales, y no demuestran relaciones causales. 

In [16]:
pred_train_lr = mejor_modelo_lr.transform(train)

metricas_train_lr = calcular_metricas(pred_train_lr)

comparacion_ajuste = pd.DataFrame([
    {
        "conjunto": "Entrenamiento",
        **metricas_train_lr
    },
    {
        "conjunto": "Validación",
        **metricas_mejor_lr
    }
])

display(comparacion_ajuste.round(4))

,conjunto,MAE,RMSE,R2
0,Entrenamiento,1237.7905,2245.2927,0.4032
1,Validación,1210.1375,2186.4196,0.4257


### Comparación entre entrenamiento y validación

El modelo obtuvo un R² de 0.4032 en entrenamiento y de 0.4257
en validación. Asimismo, el MAE y el RMSE fueron menores
en validación. Por tanto, no se observa la brecha de sobreajuste,
en la que el desempeño de entrenamiento supera al de validación.

La similitud entre ambos conjuntos sugiere que el desempeño moderado
no se explica por sobreajuste. 

Las diferencias entre trimestres también pueden influir en las métricas.
Posteriormente se comparará con Random Forest para evaluar si un modelo
capaz de representar relaciones no lineales logra reducir el error.

## Interpretación

In [14]:
mae_lr = metricas_mejor_lr["MAE"]
rmse_lr = metricas_mejor_lr["RMSE"]
r2_lr = metricas_mejor_lr["R2"]

rmse_ref = metricas_referencia["RMSE"]
cambio_rmse = 100 * (rmse_ref - rmse_lr) / rmse_ref

print(
    f"La configuración seleccionada fue {mejor_nombre}, "
    f"con regParam={mejor_config_lr['regParam']} y "
    f"elasticNetParam={mejor_config_lr['elasticNetParam']}, "
    "porque obtuvo el menor RMSE de validación entre las opciones probadas."
)

print(
    f"\nEl MAE fue Q{mae_lr:,.2f}. Esto significa que las predicciones "
    "se desviaron del salario observado en esa cantidad, en promedio "
    "y en términos absolutos."
)

print(
    f"\nEl RMSE fue Q{rmse_lr:,.2f}. Esta métrica da más peso "
    "a los errores grandes que el MAE."
)

if cambio_rmse > 0:
    print(
        f"\nLa regresión redujo el RMSE en {cambio_rmse:.2f}% "
        "respecto a predecir siempre la media del entrenamiento."
    )
elif cambio_rmse < 0:
    print(
        f"\nLa regresión aumentó el RMSE en {abs(cambio_rmse):.2f}% "
        "respecto a la referencia, por lo que no logró superarla "
        "según el criterio principal."
    )
else:
    print("\nLa regresión y la referencia obtuvieron el mismo RMSE.")

print(f"\nEl R² de validación fue {r2_lr:.4f}.")

if r2_lr >= 0:
    print(
        f"Esto corresponde a una reducción de {100 * r2_lr:.2f}% "
        "en la suma de errores cuadrados respecto a utilizar "
        "la media del salario del propio conjunto de validación."
    )
else:
    print(
        "El valor negativo indica que la suma de errores cuadrados "
        "superó la obtenida al utilizar la media del salario de validación."
    )

print(
    "\nEstos resultados corresponden al cuarto trimestre de 2025. "
    "La evaluación final en 2026 permanece pendiente. "
    "Además, la rotación de la encuesta permite que algunas personas "
    "aparezcan en distintos trimestres; la separación temporal "
    "no garantiza personas distintas entre entrenamiento y validación."
)

La configuración seleccionada fue Sin regularización, con regParam=0.0 y elasticNetParam=0.0, porque obtuvo el menor RMSE de validación entre las opciones probadas.

El MAE fue Q1,210.14. Esto significa que las predicciones se desviaron del salario observado en esa cantidad, en promedio y en términos absolutos.

El RMSE fue Q2,186.42. Esta métrica da más peso a los errores grandes que el MAE.

La regresión redujo el RMSE en 24.33% respecto a predecir siempre la media del entrenamiento.

El R² de validación fue 0.4257.
Esto corresponde a una reducción de 42.57% en la suma de errores cuadrados respecto a utilizar la media del salario del propio conjunto de validación.

Estos resultados corresponden al cuarto trimestre de 2025. La evaluación final en 2026 permanece pendiente. Además, la rotación de la encuesta permite que algunas personas aparezcan en distintos trimestres; la separación temporal no garantiza personas distintas entre entrenamiento y validación.


## Guardar el mejor pipeline

In [15]:
MODELOS_DIR = "../working_dir/modelos"
os.makedirs(MODELOS_DIR, exist_ok=True)

RUTA_MODELO_LR = f"{MODELOS_DIR}/lr_mejor_validacion"

mejor_modelo_lr.write().overwrite().save(RUTA_MODELO_LR)

resumen_lr = {
    "configuracion": mejor_config_lr,
    "periodos_entrenamiento": ["2025T1", "2025T2", "2025T3"],
    "periodo_validacion": "2025T4",
    "predictores_numericos": NUMERICAS,
    "predictores_categoricos": CATEGORICAS,
    "objetivo": OBJETIVO,
    "media_entrenamiento": float(media_train),
    "metricas_validacion_lr": metricas_mejor_lr,
    "metricas_validacion_referencia": metricas_referencia,
    "version_spark": spark.version
}

with open(
    f"{MODELOS_DIR}/lr_resumen_validacion.json",
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(resumen_lr, archivo, ensure_ascii=False, indent=2)

tabla_lr.to_csv(
    f"{MODELOS_DIR}/lr_configuraciones_validacion.csv",
    index=False
)

print("Pipeline guardado en:", RUTA_MODELO_LR)
print("Configuración y métricas guardadas.")

Pipeline guardado en: ../working_dir/modelos/lr_mejor_validacion
Configuración y métricas guardadas.
